<a href="https://colab.research.google.com/github/RajeevTricel/gemma-seo-agent/blob/main/notebooks/01_finetune_gemma_qlora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

In [ ]:
!git clone https://github.com/RajeevTricel/gemma-seo-agent.git
%cd gemma-seo-agent

In [ ]:
!ls

In [ ]:
!ls data

In [ ]:
!head -n 3 data/train.jsonl

In [ ]:
import torch
import transformers
import datasets
import peft
import trl
import bitsandbytes

print("Imports working")

In [ ]:
!pip install -U transformers datasets accelerate evaluate bitsandbytes trl peft protobuf sentencepiece huggingface_hub

In [ ]:
!pip install -U torch tensorboard
!pip install -U transformers datasets accelerate evaluate bitsandbytes trl peft protobuf sentencepiece huggingface_hub

In [ ]:
from huggingface_hub import login

login()

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="data/train.jsonl", split="train")

print(dataset)
print(dataset[0])

In [ ]:
for i, sample in enumerate(dataset):
    print(f"\nExample {i+1}")
    for msg in sample["messages"]:
        print(msg["role"], ":", msg["content"][:120])

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "google/gemma-4-E2B-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.config.use_cache = False

print("Model loaded successfully with FP16:", MODEL_ID)

In [ ]:
import torch
import transformers

print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("Transformers:", transformers.__version__)
print("GPU available:", torch.cuda.is_available())

In [ ]:
messages = [
    {
        "role": "system",
        "content": "You are a practical SEO and web performance assistant. Give clear priority-based fixes."
    },
    {
        "role": "user",
        "content": "LCP is 5.8s and the hero image is 4MB. What should I fix first?"
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=250,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

def formatting_func(example):
    return tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)

training_args = SFTConfig(
    output_dir="outputs/gemma4-e2b-seo-lora-v3-300",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    num_train_epochs=1,
    learning_rate=2e-4,
    logging_steps=10,
    save_steps=50,
    report_to="none",
    fp16=False,
    bf16=False,
    optim="paged_adamw_8bit",
    max_length=1024,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    peft_config=peft_config,
    formatting_func=formatting_func,
    processing_class=tokenizer,
)

trainer.train()

In [ ]:
trainer.model.save_pretrained("outputs/gemma4-e2b-seo-lora-adapter")
tokenizer.save_pretrained("outputs/gemma4-e2b-seo-lora-adapter")

print("Saved LoRA adapter successfully")

In [ ]:
!ls outputs/gemma4-e2b-seo-lora-adapter

In [ ]:
test_messages = [
    {
        "role": "system",
        "content": "You are a practical SEO and web performance assistant. Give clear priority-based fixes."
    },
    {
        "role": "user",
        "content": "PageSpeed score dropped badly. LCP is 6.2s, unused JS is 900KB, and the hero image is 3.8MB. What should I fix first?"
    }
]

test_prompt = tokenizer.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(test_prompt, return_tensors="pt").to(trainer.model.device)

with torch.no_grad():
    outputs = trainer.model.generate(
        **inputs,
        max_new_tokens=250,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

In [ ]:
import re
import torch

def normalize_headings(text):
    text = text.strip()

    # Normalize heading variants
    replacements = {
        r"Diagnosis\s*:": "Diagnosis:",
        r"Evidence\s*:": "Evidence:",
        r"Priority\s*:": "Priority:",
        r"Fix\s*:": "Fix:",
        r"Next action\s*:": "Next action:",
        r"Next action\s*\n": "Next action:\n",
    }

    for pattern, replacement in replacements.items():
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)

    return text


def clean_first_complete_answer(text):
    text = normalize_headings(text)

    # Cut repeated assistant/system continuations
    for marker in [
    "\nsystem\n",
    "\nmodel\n",
    "\ntype\n",
    "\nevidence\n",
    "\npriority\n",
    "\nfix\n",
    "\nnext action\n",
    "<start_of_turn>model",
    "<end_of_turn>"
]:
        if marker in text:
            text = text.split(marker)[0].strip()

    # If Diagnosis repeats, keep first answer only
    second_diag = text.find("\nDiagnosis:", 5)
    if second_diag != -1:
        text = text[:second_diag].strip()

    return normalize_headings(text)


def test_model_template_forced(user_prompt):
    test_messages = [
        {
            "role": "system",
            "content": """You are an evidence-led SEO and marketing assistant.
You must complete every section of the required template.
Do not stop early.
Do not add extra sections.
Keep each section short and practical."""
        },
        {
            "role": "user",
            "content": f"""{user_prompt}

Complete this exact template and stop after Next action:

Diagnosis:
Evidence:
Priority:
Fix:
Next action:"""
        }
    ]

    test_prompt = tokenizer.apply_chat_template(
        test_messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(test_prompt, return_tensors="pt").to(trainer.model.device)

    with torch.no_grad():
        outputs = trainer.model.generate(
            **inputs,
            max_new_tokens=350,
            do_sample=False,
            repetition_penalty=1.25,
            no_repeat_ngram_size=5,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    raw_response = tokenizer.decode(new_tokens, skip_special_tokens=True)
    response = clean_first_complete_answer(raw_response)

    print(response)

    print("\n--- Heading check ---")
    for heading in ["Diagnosis:", "Evidence:", "Priority:", "Fix:", "Next action:"]:
        print(heading, "✅" if heading in response else "❌")

In [ ]:
test_model_clean("The PageSpeed score is low but conversions are up. Should we still fix it?")

In [ ]:
test_model_clean("Clicks dropped 30%, impressions stayed almost the same, CTR dropped, and average position changed from 3.8 to 4.1. What does this mean?")

In [ ]:
test_model_final_clean("We don’t have GSC access. Diagnose the ranking drop anyway.")

In [ ]:
test_model_template_forced("Can we generate 5,000 location pages with the same copy?")

In [ ]:
test_model_template_forced("The crawl says 20 important pages are noindex. What should we do first?")

In [ ]:
trainer.model.save_pretrained("outputs/gemma4-e2b-seo-lora-v3-300")
tokenizer.save_pretrained("outputs/gemma4-e2b-seo-lora-v3-300")

print("Saved v3 300-example adapter")

In [ ]:
from huggingface_hub import login, create_repo

login()

repo_id = "RajeevSK25/gemma4-e2b-seo-lora-v3-300"

create_repo(repo_id, private=True, exist_ok=True, repo_type="model")

trainer.model.push_to_hub(repo_id)
tokenizer.push_to_hub(repo_id)

print("Pushed v3 adapter to Hugging Face:", repo_id)

In [ ]:
repo_id = "RajeevSK25/gemma4-e2b-seo-lora"

trainer.model.push_to_hub(repo_id)
tokenizer.push_to_hub(repo_id)

print("Pushed adapter to Hugging Face:", repo_id)

In [ ]:
from huggingface_hub import login, whoami

login()

print(whoami())

## Progress checkpoint

- GitHub repo created
- Dataset created in `data/train.jsonl`
- Colab connected to GitHub repo
- GPU runtime working
- `google/gemma-4-E2B-it` loaded successfully in 4-bit
- Base model test worked
- First QLoRA/LoRA training run completed successfully
- Training ran for 3 steps on 3 examples
- Tiny adapter generated bad repeated output, which is expected because the dataset is too small
- Next step: expand dataset to 50–100+ examples before serious fine-tuning